# Lab L1.3 · Transaction feature factory — 1.5M card transactions into customer features (and images into numbers)
**Horizon Gulf Bank · Project MIZAN** · CAIP AIP-210 · Certified AI Practitioner · Day 1 · ⏱ 50 min · timetabled Day 2, 15:40 · steps 1–5 & 7 (step 6 optional) · **Level 1 · From scratch**

Load 1.5M transactions memory-efficiently, aggregate them into a 250k-row customer feature table, pick the features that carry signal for income estimation, and turn unstructured document images into structured features.

Follow **L1.3_Lab_Guide.html** step by step. This notebook gives you only the structure — every line of logic is yours. Keep variable names as suggested in the guide so later steps (and later labs) line up.

In [ ]:
# ── Project MIZAN course setup ─ run this cell first ──────────────────────────
from pathlib import Path
import warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the course root (the folder that contains data/generate_data.py) from wherever this notebook lives
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "generate_data.py").exists())
DATA = ROOT / "data"          # raw synthetic datasets (read-only)
CKPT = ROOT / "checkpoints"   # hand-off artefacts between labs (lets you start any lab fresh)
CKPT.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": .3})
RNG = 42
print("Course root:", ROOT)

## Step 1 · Load 1.5M rows the memory-smart way

1. Read the file naively; record load time and `memory_usage(deep=True)` in MB; delete it.
2. Read again with `usecols` + a `dtype` map (`category` for ids and labels, `float32` for amount, `int8` for the flag) and `parse_dates`.
3. Print both timings and memory figures; save to `CKPT/card_transactions.parquet`, reload it and compare file sizes and dtypes.

In [ ]:
# Your code for step 1


## Step 2 · Aggregate events into one row per customer

1. Drop fraud rows. Add `log_amt`, `is_intl` (country ≠ AE), `is_night` (00–05h), `is_ecom` (channel E-Commerce).
2. Group by `customer_id` (`observed=True`) with named aggregation: count, total, median, mean/std of `log_amt`, the three shares, distinct MCCs and countries, first and last timestamp.
3. Add `amt_p90` (90th percentile), `recency_days` = days from the last transaction to `AS_OF`, and `txn_per_month` over the active span.

In [ ]:
# Your code for step 2


## Step 3 · Spend mix and trend features

1. Pivot spend by `customer_id × merchant_category` (sum, fill 0), divide each row by its total → `mix_<category>` shares.
2. Compute spend in the first 60 days (Apr–May) and the last 60 days (Aug–Sep); `spend_trend = log((recent+1)/(early+1))`.
3. Join both onto `agg` and show the five customers with the highest `mix_Luxury_Goods` — first for everyone, then only for customers with ≥ 10 transactions.

In [ ]:
# Your code for step 3


## Step 4 · Join to the customer master — missing means what?

1. Left-join `feat` onto `customers_clean[['customer_id']]`; assert the row count is unchanged and ids are unique.
2. Create `has_card_activity`. Fill counts, totals, mix shares and `n_*` with 0; fill averages/shares with the card-holder median; set `recency_days` to 999 for non-users.
3. Print the share of customers with card activity and confirm there are no NaNs left.

In [ ]:
# Your code for step 4


## Step 5 · Feature selection with a random probe

1. Training population: card-active customers with `salary_transfer == 1` and `salary_missing == 0`; target `y = log(monthly_salary_aed)`.
2. Add `probe_noise` (standard normal). Drop zero-variance columns with `VarianceThreshold`.
3. Find feature pairs with |Pearson r| > 0.9 and drop the second of each pair.
4. Compute `mutual_info_regression` on a 20k sample; keep features scoring above the probe. Plot the ranking.

In [ ]:
# Your code for step 5


## Step 6 · Unstructured data → structured features

1. Load the 15k KYC images; compute per image: mean ink, ink std, horizontal and vertical edge energy, ink in the top/bottom/left/right halves, number of ‘text rows’ (rows whose ink exceeds the image mean).
2. Fit a quick multinomial `LogisticRegression` on 80% / test on 20% and print accuracy against the 20% majority baseline.

In [ ]:
# Your code for step 6


## Step 7 · Persist the feature table and its contract

1. Save `X` to `CKPT/customer_txn_features.parquet`.
2. Write `txn_feature_spec.json` with `as_of`, `selected`, `dropped_correlated` and the MI scores.

In [ ]:
# Your code for step 7
